# Least Squares and Residuals

In [ ]:
library(tidyverse)
options(repr.matrix.max.rows = 20)

## Review: The Regression Estimate

$$\text{estimate of } y = r \times x \quad \text{(standard units)}$$

First round: average 70, SD 10. Final round: average 50, SD 12. $r = 0.75$. Predict the final round score for a first round score of 90.

In [ ]:
average_x <- 70
sd_x <- 10
average_y <- 50
sd_y <- 12
r <- 0.75

Step 1: convert to standard units.

In [ ]:
x_su <- (90 - average_x) / sd_x
x_su

Step 2: multiply by $r$.

In [ ]:
y_su <- r * x_su
y_su

Step 3: convert back to original units.

In [ ]:
y_su * sd_y + average_y

$$\text{slope} = r \times \frac{\text{SD of } y}{\text{SD of } x}$$

$$\text{intercept} = \text{average of } y - \text{slope} \times \text{average of } x$$

In [ ]:
slope <- r * sd_y / sd_x
intercept <- average_y - slope * average_x
c(slope, intercept)

In [ ]:
slope * 90 + intercept

What about a first round score of 60?

In [ ]:
slope * 60 + intercept

## Part 1: Error in Estimation

In [ ]:
demographics <- read_csv("data/district_demographics2016.csv", show_col_types = FALSE)
demographics

In [ ]:
demographics <- demographics |>
  select(`Median Income`, `College%`)

demographics

In [ ]:
ggplot(demographics, aes(x = `College%`, y = `Median Income`)) +
  geom_point(alpha = 0.5) +
  theme_minimal(base_size = 16)

Guess $r$.

In [ ]:
demographics_r <- cor(demographics$`College%`, demographics$`Median Income`)
demographics_r

In [ ]:
demographics_slope <- demographics_r * sd(demographics$`Median Income`) / sd(demographics$`College%`)
demographics_intercept <- mean(demographics$`Median Income`) - demographics_slope * mean(demographics$`College%`)
c(demographics_slope, demographics_intercept)

In [ ]:
demographics_model <- lm(`Median Income` ~ `College%`, data = demographics)
coef(demographics_model)

`fitted()`: the line's estimate for every row.

In [ ]:
demographics <- demographics |>
  mutate(`Linear Prediction` = fitted(demographics_model))

ggplot(demographics, aes(x = `College%`)) +
  geom_point(aes(y = `Median Income`), alpha = 0.5) +
  geom_point(aes(y = `Linear Prediction`), color = "goldenrod", size = 2) +
  labs(y = "Median Income") +
  theme_minimal(base_size = 16)

$$\text{error} = \text{actual value} - \text{estimate}$$

In [ ]:
demographics <- demographics |>
  mutate(Error = `Median Income` - `Linear Prediction`)

demographics

The average error:

In [ ]:
mean(demographics$Error)

Square, take the mean, take the square root: the **root mean squared error (rmse)**.

In [ ]:
sqrt(mean(demographics$Error^2))

### Other Lines

In [ ]:
sample_points <- tibble(
  x = c(14.7, 19.1, 50.7, 59.5),
  y = c(33995, 61454, 71183, 105918)
)

demographics_errors <- function(slope, intercept) {
  ggplot(demographics, aes(x = `College%`, y = `Median Income`)) +
    geom_point(alpha = 0.5) +
    geom_abline(slope = slope, intercept = intercept,
                color = "dodgerblue3", linewidth = 1.2) +
    geom_segment(data = sample_points,
                 aes(x = x, xend = x, y = y, yend = slope * x + intercept),
                 color = "red", linewidth = 1.2) +
    theme_minimal(base_size = 16)
}

The regression line:

In [ ]:
demographics_errors(demographics_slope, demographics_intercept)

A different line:

In [ ]:
demographics_errors(1500, 20000)

A silly line:

In [ ]:
demographics_errors(-1000, 75000)

### Root Mean Squared Error

In [ ]:
demographics_rmse <- function(slope, intercept) {
  x <- demographics$`College%`
  y <- demographics$`Median Income`
  fitted <- slope * x + intercept
  mse <- mean((y - fitted)^2)
  cat("Root mean squared error:", sqrt(mse), "\n")
  demographics_errors(slope, intercept)
}

In [ ]:
demographics_rmse(-1000, 75000)

In [ ]:
demographics_rmse(1500, 20000)

In [ ]:
demographics_rmse(demographics_slope, demographics_intercept)

Is there a line with a smaller rmse?

## Part 2: Numerical Optimization

In [ ]:
f <- function(x) {
  (x - 2)^2 + 3
}

In [ ]:
tibble(x = seq(1, 3, by = 0.1), y = f(x)) |>
  ggplot(aes(x, y)) +
  geom_line(color = "dodgerblue3", linewidth = 1.2) +
  theme_minimal(base_size = 16)

`optim(start, function)`. `$par` is the input it found.

In [ ]:
optim(0, f, method = "BFGS")$par

In [ ]:
complicated_function <- function(x) {
  2 * sin(x * pi) + x^3 + x^4
}

In [ ]:
tibble(x = seq(-1.5, 1.5, by = 0.05), y = complicated_function(x)) |>
  ggplot(aes(x, y)) +
  geom_line(color = "dodgerblue3", linewidth = 1.2) +
  theme_minimal(base_size = 16)

In [ ]:
optim(0, complicated_function, method = "BFGS")$par

### Minimizing the Mean Squared Error

`params[1]` is the slope and `params[2]` is the intercept.

In [ ]:
demographics_mse <- function(params) {
  any_slope <- params[1]
  any_intercept <- params[2]
  x <- demographics$`College%`
  y <- demographics$`Median Income`
  fitted <- any_slope * x + any_intercept
  mean((y - fitted)^2)
}

Check: the square root is the rmse.

In [ ]:
sqrt(demographics_mse(c(1500, 20000)))

In [ ]:
sqrt(demographics_mse(c(-1000, 75000)))

Search over all lines, starting at slope 0 and intercept 0.

In [ ]:
best <- optim(c(0, 0), demographics_mse)$par
best

The slope and intercept from the formulas:

In [ ]:
c(demographics_slope, demographics_intercept)

The regression line is the **least squares line**.

## Part 3: Nonlinear Regression

In [ ]:
shotput <- read_csv("data/shotput.csv", show_col_types = FALSE)
shotput

In [ ]:
ggplot(shotput, aes(x = `Weight Lifted`, y = `Shot Put Distance`)) +
  geom_point() +
  theme_minimal(base_size = 16)

The best straight line, from `lm()` and from `optim()`:

In [ ]:
shotput_model <- lm(`Shot Put Distance` ~ `Weight Lifted`, data = shotput)
coef(shotput_model)

In [ ]:
shotput_linear_mse <- function(params) {
  any_slope <- params[1]
  any_intercept <- params[2]
  x <- shotput$`Weight Lifted`
  y <- shotput$`Shot Put Distance`
  fitted <- any_slope * x + any_intercept
  mean((y - fitted)^2)
}

optim(c(0, 0), shotput_linear_mse)$par

In [ ]:
shotput |>
  mutate(`Best Straight Line` = fitted(shotput_model)) |>
  ggplot(aes(x = `Weight Lifted`)) +
  geom_point(aes(y = `Shot Put Distance`)) +
  geom_point(aes(y = `Best Straight Line`), color = "goldenrod", size = 2) +
  labs(y = "Shot Put Distance") +
  theme_minimal(base_size = 16)

**Quadratic function**

$$f(x) = ax^2 + bx + c$$

In [ ]:
shotput_quadratic_mse <- function(params) {
  a <- params[1]
  b <- params[2]
  c <- params[3]
  x <- shotput$`Weight Lifted`
  y <- shotput$`Shot Put Distance`
  fitted <- a * x^2 + b * x + c
  mean((y - fitted)^2)
}

In [ ]:
best <- optim(c(0, 0, 0), shotput_quadratic_mse)$par
best

Predict for 100 kilograms:

In [ ]:
best[1] * 100^2 + best[2] * 100 + best[3]

In [ ]:
shotput |>
  mutate(`Best Quadratic Curve` = best[1] * `Weight Lifted`^2 +
           best[2] * `Weight Lifted` + best[3]) |>
  ggplot(aes(x = `Weight Lifted`)) +
  geom_point(aes(y = `Shot Put Distance`)) +
  geom_point(aes(y = `Best Quadratic Curve`), color = "goldenrod", size = 2) +
  labs(y = "Shot Put Distance") +
  theme_minimal(base_size = 16)

## Part 4: Residuals

$$\text{residual} = \text{observed value} - \text{regression estimate}$$

In [ ]:
original <- read_csv("data/family_heights.csv", show_col_types = FALSE)

heights <- tibble(
  MidParent = original$midparentHeight,
  Child = original$childHeight
)

family_model <- lm(Child ~ MidParent, data = heights)
coef(family_model)

In [ ]:
heights <- heights |>
  mutate(
    `Fitted Value` = fitted(family_model),
    Residual = residuals(family_model)
  )

heights

The average of the residuals:

In [ ]:
mean(heights$Residual)

`scatter_fit()`: the scatter plot with the regression line.

In [ ]:
scatter_fit <- function(table, x, y) {
  model <- lm(table[[y]] ~ table[[x]])
  ggplot(table, aes(x = .data[[x]], y = .data[[y]])) +
    geom_point(alpha = 0.5) +
    geom_abline(intercept = coef(model)[1], slope = coef(model)[2],
                color = "gold", linewidth = 2) +
    labs(x = x, y = y) +
    theme_minimal(base_size = 16)
}

In [ ]:
scatter_fit(heights, "MidParent", "Child")

`residual_plot()`: the residuals against $x$.

In [ ]:
residual_plot <- function(table, x, y) {
  model <- lm(table[[y]] ~ table[[x]])
  tibble(x_values = table[[x]], residuals = residuals(model)) |>
    ggplot(aes(x = x_values, y = residuals)) +
    geom_point(color = "red", alpha = 0.5) +
    geom_hline(yintercept = 0, color = "darkblue", linewidth = 2) +
    labs(x = x, title = "Residual Plot") +
    theme_minimal(base_size = 16)
}

In [ ]:
residual_plot(heights, "MidParent", "Child")

No pattern. `regression_diagnostic_plots()` draws both plots.

In [ ]:
regression_diagnostic_plots <- function(table, x, y) {
  print(scatter_fit(table, x, y))
  print(residual_plot(table, x, y))
}

### Detecting Nonlinearity

Dugongs: predict `Age` (years) from `Length` (meters).

In [ ]:
dugong <- read_csv("data/dugongs.csv", show_col_types = FALSE) |>
  select(Length, Age)

dugong

In [ ]:
cor(dugong$Length, dugong$Age)

In [ ]:
regression_diagnostic_plots(dugong, "Length", "Age")

**When a residual plot shows a pattern, there may be a nonlinear relation between the variables.**

### Detecting Uneven Spread

In [ ]:
regression_diagnostic_plots(demographics, "College%", "Median Income")

**If the residual plot shows uneven spread around the line at 0, the regression estimates are not equally accurate across the range of $x$.**

## Try it out!

Last class you fit a line to predict a penguin's body mass (`body_mass_g`, in grams) from its flipper length (`flipper_length_mm`). How well does it fit?

1. Add two columns to `model_data`: the fitted values and the residuals.
2. Find the average of the residuals.
3. Find the rmse. What are its units?
4. Draw the residual plot with `residual_plot()`. Do you see a pattern?

**Done early?** Which penguin has the largest residual?

In [ ]:
library(palmerpenguins)

# Keep rows with the two variables we need
model_data <- penguins |>
  filter(!is.na(flipper_length_mm), !is.na(body_mass_g)) |>
  select(species, flipper_length_mm, body_mass_g)

penguin_model <- lm(body_mass_g ~ flipper_length_mm, data = model_data)
coef(penguin_model)

In [ ]:
# Your code here

In [ ]:
# Your code here

In [ ]:
# Your code here

In [ ]:
# Done early? The largest residual